# OncoBridge Stage 2 — Full Architecture Ablation Suite (v2)

**Does NOT retrain the full drug-response model.** Only PCC / Avg Drug
Pearson / RMSE / R\u00b2 are reported anywhere — no AUROC, no drug-cold.

**Training budget cut from your original ~90 min/run to keep 8 trained
ablations tractable in one sitting:**

| | Original | This notebook |
|---|---|---|
| `stage1_epochs` / `patience` | 25 / 7 | **8 / 3** |
| `stage2_epochs` / `patience` | 30 / 10 | **10 / 4** |
| Max total epochs | 55 | **18** (~33%) |

This is a real tradeoff — these ablations are directional, not fully
converged. Compare the 8 trained ablations **to each other** (all share the
same reduced budget) rather than to the full-budget reference checkpoint.
Morgan-only skips the ChemBERTa transformer forward pass entirely, so it
will run noticeably faster than the other 7.

| # | Ablation | Needs training? | What it tests |
|---|---|---|---|
| 0 | Full model (loaded checkpoint, or manuscript numbers) | No | Reference |
| 1 | Drug-identity lesion | No \u2014 forward pass | Drug-mean recovery share of PCC |
| 2 | Cell-line lesion | No \u2014 forward pass | Cell-line-specific signal share |
| 3 | Modality zero-out (\u00d74) | No \u2014 forward pass | Which omics layer matters at inference |
| 4 | Concat Fusion | **Yes** | Cross-attention vs concat+MLP |
| 5 | Gene-Gating Disabled | **Yes** | Does the learned gate matter? |
| 6 | Phase-1-only | **Yes** (cheap) | Real contribution of Phase 2 |
| 7 | ChemBERTa-only drug encoder | **Yes** | Value of the Morgan FP branch |
| 8 | Morgan-only drug encoder | **Yes** (cheap \u2014 no ChemBERTa) | Value of the ChemBERTa branch |
| 9 | Few Fusion-Heads (8\u21922) | **Yes** | Is fusion attention over-provisioned? |
| 10 | Unfreeze-Depth=4 (was 2) | **Yes** | Is 2 unfrozen layers the right amount? |
| 11 | Linear Regression Head | **Yes** | Does the 3-layer regression head matter? |

Everything from #4 onward is built from ONE flexible model class
(`OncoBridgeDrugResponseFlexible`) with swappable drug-encoder / fusion /
regression-head components \u2014 no per-ablation model class duplication.

In [1]:
# ═══════════════════════════════════════════════════════════════════════
#  CELL 1 — INSTALLS, IMPORTS, CONFIG
# ═══════════════════════════════════════════════════════════════════════
import subprocess, sys
def pip(*pkgs): subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=True)
pip('transformers', 'accelerate')

import os, gc, json, math, time, random, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint as grad_checkpoint
from transformers import AutoTokenizer, AutoModel
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_squared_error, r2_score

def section(t):
    print('\n' + '='*70); print(f'  {t}'); print('='*70)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_GPUS = torch.cuda.device_count()
print(f'Device: {DEVICE} | GPUs: {N_GPUS}')

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

SPLIT_MODE = 'cell_cold'   # 'warm' | 'cell_cold' | 'drug_cold'

DATA_BASE = '/kaggle/input/datasets/proutkarshtiwari/smiles2-training-data'
CKPT_PATH = '/kaggle/input/models/proutkarshtiwari/oncobridgev7-95-74acc/pytorch/default/1/oncobridge_v7_best.pt'
OUT_DIR   = '/kaggle/working/'
EXISTING_STAGE2_CKPT_PATH = f'{OUT_DIR}oncobridge_chemberta_v5_best.pt'

PATH_MRNA       = f'{DATA_BASE}/ccle_mrna_scaled.parquet'
PATH_CNV        = f'{DATA_BASE}/ccle_cnv_scaled.parquet'
PATH_MUT        = f'{DATA_BASE}/ccle_mut_scaled.parquet'
PATH_METH       = f'{DATA_BASE}/ccle_meth_scaled.parquet'
PATH_SMILES     = f'{DATA_BASE}/drug_smiles.json'
PATH_DRUG_NAMES = f'{DATA_BASE}/valid_drug_names.csv'
PATH_DRUG2IDX   = f'{DATA_BASE}/drug2idx.json'

SPLIT_PATHS = {
    'warm':      {'train': f'{DATA_BASE}/split_warm_tr.parquet', 'val': f'{DATA_BASE}/split_warm_vl.parquet', 'test': f'{DATA_BASE}/split_warm_te.parquet'},
    'cell_cold': {'train': f'{DATA_BASE}/split_cc_tr.parquet',   'val': f'{DATA_BASE}/split_cc_vl.parquet',   'test': f'{DATA_BASE}/split_cc_te.parquet'},
    'drug_cold': {'train': f'{DATA_BASE}/split_dc_tr.parquet',   'val': f'{DATA_BASE}/split_dc_vl.parquet',   'test': f'{DATA_BASE}/split_dc_te.parquet'},
}

CONFIG = {
    'num_mrna_genes': 8011, 'num_cnv_genes': 3500, 'num_mut_genes': 2500, 'num_meth_genes': 6000,
    'embed_dim': 384, 'num_heads': 8, 'num_encoder_layers': 6, 'num_cross_layers': 4,
    'cnn_kernel': 16, 'cnn_stride': 16, 'ff_dim': 1536, 'dropout': 0.20, 'gate_init': 1.5,
    'use_grad_ckpt': False, 'num_classes': 22,
    'chemberta_model': 'DeepChem/ChemBERTa-77M-MLM', 'chemberta_dim': 384,
    'chemberta_max_len': 512, 'chemberta_freeze_layers': 6,
    'morgan_fp_path': f'{DATA_BASE}/drug_morgan_fps.npy',
    'morgan_fp_order_path': f'{DATA_BASE}/drug_morgan_fp_order.json', 'morgan_dim': 1024,
    'fusion_heads': 8, 'fusion_dropout': 0.20,
    'batch_size': 32, 'grad_accum': 4, 'num_workers': 4, 'use_amp': True, 'clip_grad': 1.0,
    'warmup_frac': 0.10,
    # ── Reduced budget so 8 trained ablations are tractable in one sitting ──
    'stage1_epochs': 8, 'stage1_lr': 5e-4, 'stage1_wd': 1e-4, 'stage1_patience': 3,
    'stage2_epochs': 10, 'stage2_lr_head': 1e-4, 'stage2_lr_cross': 5e-5,
    'stage2_lr_cbert': 2e-5, 'stage2_lr_proj': 1e-4, 'stage2_lr_mod_enc': 5e-6,
    'stage2_unfreeze_mod_layers': 2, 'stage2_wd': 1e-4, 'stage2_patience': 4,
    'checkpoint_ft': f'{OUT_DIR}oncobridge_chemberta_v5_best.pt',
    'split_mode': SPLIT_MODE,
}
print(f'SPLIT_MODE={SPLIT_MODE} | Reduced budget: '
      f'stage1={CONFIG["stage1_epochs"]}ep/{CONFIG["stage1_patience"]}pat, '
      f'stage2={CONFIG["stage2_epochs"]}ep/{CONFIG["stage2_patience"]}pat')
print('This notebook does NOT retrain the full-budget model.')

Device: cuda | GPUs: 2
SPLIT_MODE=cell_cold | Reduced budget: stage1=8ep/3pat, stage2=10ep/4pat
This notebook does NOT retrain the full-budget model.


In [2]:
# ═══════════════════════════════════════════════════════════════════════
#  CELL 2 — LOAD PROCESSED DATA & SPLITS
# ═══════════════════════════════════════════════════════════════════════
section('2. LOAD PROCESSED DATA')
mrna_df = pd.read_parquet(PATH_MRNA); cnv_df = pd.read_parquet(PATH_CNV)
mut_df  = pd.read_parquet(PATH_MUT);  meth_df = pd.read_parquet(PATH_METH)

with open(PATH_SMILES)   as f: smiles_map = json.load(f)
with open(PATH_DRUG2IDX) as f: drug2idx   = json.load(f)
DRUG_NAMES = pd.read_csv(PATH_DRUG_NAMES, header=None)[0].tolist()
id_to_drug = {int(i): d for d, i in drug2idx.items()}

morgan_fp_matrix = np.load(CONFIG['morgan_fp_path']).astype(np.float32)
with open(CONFIG['morgan_fp_order_path']) as f: morgan_fp_order = json.load(f)
drug_to_morgan_idx = {d: i for i, d in enumerate(morgan_fp_order)}

all_cell_lines = sorted(mrna_df.index.tolist())
cl_to_idx = {cl: i for i, cl in enumerate(all_cell_lines)}
mrna_arr = mrna_df.loc[all_cell_lines].values.astype(np.float32)
cnv_arr  = cnv_df.loc[all_cell_lines].values.astype(np.float32)
mut_arr  = mut_df.loc[all_cell_lines].values.astype(np.float32)
meth_arr = meth_df.loc[all_cell_lines].values.astype(np.float32)

def load_and_map_split(path):
    df = pd.read_parquet(path)
    df['cl_idx']   = df['ModelID'].map(cl_to_idx)
    df['drug_idx'] = df['DRUG_NAME'].map(drug2idx)
    df = df.dropna(subset=['cl_idx', 'drug_idx'])
    df['cl_idx']   = df['cl_idx'].astype(int)
    df['drug_idx'] = df['drug_idx'].astype(int)
    return df

ic50_train = load_and_map_split(SPLIT_PATHS[SPLIT_MODE]['train'])
ic50_val   = load_and_map_split(SPLIT_PATHS[SPLIT_MODE]['val'])
ic50_test  = load_and_map_split(SPLIT_PATHS[SPLIT_MODE]['test'])
print(f'  [{SPLIT_MODE}] Train: {len(ic50_train):,} | Val: {len(ic50_val):,} | Test: {len(ic50_test):,}')

del mrna_df, cnv_df, mut_df, meth_df; gc.collect()


  2. LOAD PROCESSED DATA
  [cell_cold] Train: 6,993 | Val: 833 | Test: 2,141


131

In [3]:
# ═══════════════════════════════════════════════════════════════════════
#  CELL 3 — DATASET & DATALOADERS
# ═══════════════════════════════════════════════════════════════════════
section('3. DATASET & DATALOADERS')
tokenizer = AutoTokenizer.from_pretrained(CONFIG['chemberta_model'])

class DrugResponseDataset(Dataset):
    def __init__(self, df, mrna_arr, cnv_arr, mut_arr, meth_arr, smiles_map, tokenizer,
                 morgan_fp_matrix, drug_to_morgan_idx, max_len=512):
        self.cl_idx = df['cl_idx'].values.astype(np.int32)
        self.drug_idx = df['drug_idx'].values.astype(np.int64)
        self.ic50 = df['ic50_value'].values.astype(np.float32)
        self.drug_names = df['DRUG_NAME'].values
        self.mrna_arr, self.cnv_arr = mrna_arr, cnv_arr
        self.mut_arr,  self.meth_arr = mut_arr, meth_arr
        self.smiles_map, self.tokenizer, self.max_len = smiles_map, tokenizer, max_len
        self.morgan_fp_matrix = morgan_fp_matrix
        self.drug_to_morgan_idx = drug_to_morgan_idx

    def __len__(self): return len(self.ic50)

    def __getitem__(self, idx):
        ci, did = int(self.cl_idx[idx]), int(self.drug_idx[idx])
        dname = self.drug_names[idx]
        mrna = torch.from_numpy(self.mrna_arr[ci].copy())
        cnv  = torch.from_numpy(self.cnv_arr[ci].copy())
        mut  = torch.from_numpy(self.mut_arr[ci].copy())
        meth = torch.from_numpy(self.meth_arr[ci].copy())
        smiles = self.smiles_map.get(dname, 'C') or 'C'
        enc = self.tokenizer(smiles, max_length=self.max_len, padding='max_length',
                              truncation=True, return_tensors='pt')
        fp_row = self.drug_to_morgan_idx.get(dname, 0)
        morgan_fp = torch.from_numpy(self.morgan_fp_matrix[fp_row].copy())
        return {'mrna': mrna, 'cnv': cnv, 'mut': mut, 'meth': meth,
                'input_ids': enc['input_ids'].squeeze(0),
                'attention_mask': enc['attention_mask'].squeeze(0),
                'morgan_fp': morgan_fp, 'ic50': torch.tensor(self.ic50[idx]),
                'drug_idx': torch.tensor(did)}

def make_loader(df, shuffle, bs_mult=1):
    ds = DrugResponseDataset(df, mrna_arr, cnv_arr, mut_arr, meth_arr, smiles_map,
                              tokenizer, morgan_fp_matrix, drug_to_morgan_idx,
                              CONFIG['chemberta_max_len'])
    return DataLoader(ds, batch_size=CONFIG['batch_size']*bs_mult, shuffle=shuffle,
                       num_workers=CONFIG['num_workers'], pin_memory=True,
                       persistent_workers=(CONFIG['num_workers'] > 0),
                       prefetch_factor=2 if CONFIG['num_workers'] > 0 else None,
                       drop_last=shuffle), ds

train_loader, train_ds = make_loader(ic50_train, shuffle=True)
val_loader,   val_ds   = make_loader(ic50_val,   shuffle=False, bs_mult=2)
test_loader,  test_ds  = make_loader(ic50_test,  shuffle=False, bs_mult=2)
print(f'Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')


  3. DATASET & DATALOADERS


config.json:   0%|          | 0.00/631 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/420 [00:00<?, ?B/s]

Train batches: 218 | Val: 14 | Test: 34


In [4]:
# ═══════════════════════════════════════════════════════════════════════
#  CELL 4 — MODEL ARCHITECTURE CLASSES
# ═══════════════════════════════════════════════════════════════════════
section('4. MODEL ARCHITECTURE')

class GeneImportanceLayer(nn.Module):
    def __init__(self, num_genes, init_val=1.5):
        super().__init__()
        self.logits = nn.Parameter(torch.full((num_genes,), init_val))
    def forward(self, x): return x * torch.sigmoid(self.logits)


class ModalityEncoder(nn.Module):
    def __init__(self, num_genes, embed_dim, num_heads, num_layers,
                 cnn_kernel, cnn_stride, ff_dim, dropout, gate_init=1.5, use_ckpt=False):
        super().__init__()
        self.use_ckpt = use_ckpt
        self.gene_gate = GeneImportanceLayer(num_genes, init_val=gate_init)
        self.input_proj = nn.Linear(1, embed_dim)
        self.cnn = nn.Sequential(
            nn.Conv1d(embed_dim, embed_dim, kernel_size=cnn_kernel, stride=cnn_stride,
                      padding=cnn_kernel // 2), nn.GELU(), nn.BatchNorm1d(embed_dim))
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)
        compressed = (num_genes + cnn_kernel // 2 * 2 - cnn_kernel) // cnn_stride + 1 + 1
        self.pos_emb = nn.Parameter(torch.randn(1, compressed, embed_dim) * 0.02)
        enc_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads,
            dim_feedforward=ff_dim, dropout=dropout, batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(embed_dim)
    def _tfm(self, x): return self.transformer(x)
    def forward(self, x):
        x = self.gene_gate(x).unsqueeze(-1)
        x = self.input_proj(x).transpose(1, 2)
        x = self.cnn(x).transpose(1, 2)
        B = x.size(0)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1) + self.pos_emb[:, :x.size(1) + 1, :]
        x = (grad_checkpoint(self._tfm, x, use_reentrant=False)
             if self.use_ckpt and self.training else self._tfm(x))
        return self.norm(x)


class CrossModalAttention4(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout):
        super().__init__()
        mods = ['mrna', 'cnv', 'mut', 'meth']
        self.cross_attns = nn.ModuleDict({m: nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True) for m in mods})
        self.q_norms = nn.ModuleDict({m: nn.LayerNorm(embed_dim) for m in mods})
        def ffn(): return nn.Sequential(nn.Linear(embed_dim, ff_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(ff_dim, embed_dim), nn.Dropout(dropout))
        self.ffns = nn.ModuleDict({m: ffn() for m in mods})
        self.ffn_norms = nn.ModuleDict({m: nn.LayerNorm(embed_dim) for m in mods})
    def forward(self, mrna_seq, cnv_seq, mut_seq, meth_seq):
        seqs = {'mrna': mrna_seq, 'cnv': cnv_seq, 'mut': mut_seq, 'meth': meth_seq}
        out = {}
        for m, q_seq in seqs.items():
            others = torch.cat([s for k, s in seqs.items() if k != m], dim=1)
            q = self.q_norms[m](q_seq)
            h, _ = self.cross_attns[m](q, others, others, need_weights=False)
            ao = q_seq + h
            out[m] = ao + self.ffns[m](self.ffn_norms[m](ao))
        return out['mrna'], out['cnv'], out['mut'], out['meth']


class GatedFusion4(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        self.gate = nn.Linear(embed_dim * 4, 4)
    def forward(self, m, c, u, e):
        g = F.softmax(self.gate(torch.cat([m, c, u, e], dim=-1)), dim=-1)
        return g[:, 0:1]*m + g[:, 1:2]*c + g[:, 2:3]*u + g[:, 3:4]*e, g


class OncoBridgeMMCAT_v7(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        E, H, NL, NC = cfg['embed_dim'], cfg['num_heads'], cfg['num_encoder_layers'], cfg['num_cross_layers']
        K, S, FF, D  = cfg['cnn_kernel'], cfg['cnn_stride'], cfg['ff_dim'], cfg['dropout']
        GI, UC       = cfg['gate_init'], cfg['use_grad_ckpt']
        self.mrna_enc = ModalityEncoder(cfg['num_mrna_genes'], E, H, NL, K, S, FF, D, GI, UC)
        self.cnv_enc  = ModalityEncoder(cfg['num_cnv_genes'],  E, H, NL, K, S, FF, D, GI, UC)
        self.mut_enc  = ModalityEncoder(cfg['num_mut_genes'],  E, H, NL, K, S, FF, D, GI, UC)
        self.meth_enc = ModalityEncoder(cfg['num_meth_genes'], E, H, NL, K, S, FF, D, GI, UC)
        self.cross_layers = nn.ModuleList([CrossModalAttention4(E, H, FF, D) for _ in range(NC)])
        self.fusion = GatedFusion4(E)
        self.classifier = nn.Sequential(
            nn.LayerNorm(E*5), nn.Linear(E*5, E*2), nn.GELU(), nn.Dropout(D),
            nn.Linear(E*2, E), nn.GELU(), nn.Dropout(D), nn.Linear(E, cfg['num_classes']))
    def forward(self, mrna, cnv, mut, meth):
        ms = self.mrna_enc(mrna); cs = self.cnv_enc(cnv)
        us = self.mut_enc(mut);   es = self.meth_enc(meth)
        for layer in self.cross_layers:
            ms, cs, us, es = layer(ms, cs, us, es)
        cm = ms[:, 0]; cc = cs[:, 0]; cu = us[:, 0]; ce = es[:, 0]
        fused, gates = self.fusion(cm, cc, cu, ce)
        return self.classifier(torch.cat([cm, cc, cu, ce, fused], dim=-1))


# ── Drug encoder variants (component #7/#8 ablation) ──────────────────
class ChemBERTaDrugEncoder(nn.Module):
    """Full dual-branch: ChemBERTa CLS (half) + Morgan FP (half)."""
    def __init__(self, model_name, drug_embed_dim, freeze_n_layers=6, dropout=0.1, morgan_dim=1024):
        super().__init__()
        assert drug_embed_dim % 2 == 0
        half_dim = drug_embed_dim // 2
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        for name, p in self.encoder.named_parameters():
            layer_num = None
            if 'layer.' in name:
                try: layer_num = int(name.split('layer.')[1].split('.')[0])
                except Exception: layer_num = None
            p.requires_grad = (layer_num is not None and layer_num >= freeze_n_layers)
        n_tr  = sum(p.numel() for p in self.encoder.parameters() if p.requires_grad)
        n_tot = sum(p.numel() for p in self.encoder.parameters())
        print(f'  ChemBERTa: {n_tot:,} total | {n_tr:,} trainable (layers {freeze_n_layers}+)')
        self.smiles_proj = nn.Sequential(nn.Linear(hidden_size, half_dim), nn.GELU(),
                                          nn.Dropout(dropout), nn.LayerNorm(half_dim))
        self.morgan_proj = nn.Sequential(nn.Linear(morgan_dim, half_dim), nn.GELU(),
                                          nn.Dropout(dropout), nn.LayerNorm(half_dim))
        self.fusion_proj = nn.Sequential(nn.Linear(drug_embed_dim, drug_embed_dim), nn.LayerNorm(drug_embed_dim))
    def forward(self, input_ids, attention_mask, morgan_fp):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        smiles_emb = self.smiles_proj(out.last_hidden_state[:, 0, :])
        morgan_emb = self.morgan_proj(morgan_fp)
        return self.fusion_proj(torch.cat([smiles_emb, morgan_emb], dim=-1))


class ChemBERTaOnlyDrugEncoder(nn.Module):
    """Ablation: ChemBERTa branch only, full drug_embed_dim, no Morgan FP."""
    def __init__(self, model_name, drug_embed_dim, freeze_n_layers=6, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        for name, p in self.encoder.named_parameters():
            layer_num = None
            if 'layer.' in name:
                try: layer_num = int(name.split('layer.')[1].split('.')[0])
                except Exception: layer_num = None
            p.requires_grad = (layer_num is not None and layer_num >= freeze_n_layers)
        self.smiles_proj = nn.Sequential(nn.Linear(hidden_size, drug_embed_dim), nn.GELU(),
                                          nn.Dropout(dropout), nn.LayerNorm(drug_embed_dim))
    def forward(self, input_ids, attention_mask, morgan_fp):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return self.smiles_proj(out.last_hidden_state[:, 0, :])


class MorganOnlyDrugEncoder(nn.Module):
    """Ablation: Morgan FP branch only. No ChemBERTa forward pass at all —
    the cheapest of the drug-encoder ablations to train."""
    def __init__(self, drug_embed_dim, morgan_dim=1024, dropout=0.1):
        super().__init__()
        self.morgan_proj = nn.Sequential(nn.Linear(morgan_dim, drug_embed_dim), nn.GELU(),
                                          nn.Dropout(dropout), nn.LayerNorm(drug_embed_dim))
    def forward(self, input_ids, attention_mask, morgan_fp):
        return self.morgan_proj(morgan_fp)


# ── Fusion variants ─────────────────────────────────────────────────
class DrugPatientCrossAttention(nn.Module):
    def __init__(self, patient_dim, drug_embed_dim, embed_dim, num_heads, dropout, output_dim):
        super().__init__()
        assert patient_dim % embed_dim == 0
        self.n_patient_tokens = patient_dim // embed_dim
        self.embed_dim = embed_dim
        self.drug_proj = nn.Sequential(nn.Linear(drug_embed_dim, embed_dim), nn.LayerNorm(embed_dim))
        self.cross_attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.attn_norm = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(nn.Linear(embed_dim, embed_dim*2), nn.GELU(), nn.Dropout(dropout),
                                  nn.Linear(embed_dim*2, embed_dim), nn.Dropout(dropout))
        self.ffn_norm = nn.LayerNorm(embed_dim)
        combined = embed_dim + drug_embed_dim
        self.output_mlp = nn.Sequential(nn.Linear(combined, combined//2), nn.GELU(), nn.Dropout(dropout),
                                         nn.Linear(combined//2, output_dim), nn.LayerNorm(output_dim))
    def forward(self, patient_embed, drug_embed):
        B = patient_embed.size(0)
        patient_tokens = patient_embed.view(B, self.n_patient_tokens, self.embed_dim)
        drug_query = self.drug_proj(drug_embed).unsqueeze(1)
        attn_out, attn_w = self.cross_attn(drug_query, patient_tokens, patient_tokens, need_weights=True)
        attn_out = self.attn_norm(drug_query + attn_out)
        ffn_out  = self.ffn_norm(attn_out + self.ffn(attn_out))
        combined = torch.cat([ffn_out.squeeze(1), drug_embed], dim=-1)
        return self.output_mlp(combined), attn_w


class DrugPatientConcatFusion(nn.Module):
    """Ablation: concat(patient_embed, drug_embed) -> MLP instead of cross-attention."""
    def __init__(self, patient_dim, drug_embed_dim, output_dim, dropout):
        super().__init__()
        combined = patient_dim + drug_embed_dim
        self.mlp = nn.Sequential(
            nn.LayerNorm(combined), nn.Linear(combined, combined // 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(combined // 2, output_dim), nn.LayerNorm(output_dim))
    def forward(self, patient_embed, drug_embed):
        return self.mlp(torch.cat([patient_embed, drug_embed], dim=-1)), None


# ── Regression head variants ────────────────────────────────────────
class FullRegressionHead(nn.Module):
    def __init__(self, fusion_out_dim, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(fusion_out_dim), nn.Linear(fusion_out_dim, 256), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, 64), nn.GELU(), nn.Dropout(dropout), nn.Linear(64, 1))
    def forward(self, x): return self.net(x)


class LinearRegressionHead(nn.Module):
    """Ablation: single linear layer, no depth at all."""
    def __init__(self, fusion_out_dim, dropout=None):
        super().__init__()
        self.net = nn.Linear(fusion_out_dim, 1)
    def forward(self, x): return self.net(x)


# ── ONE flexible model class used by every trained ablation below ─────
class OncoBridgeDrugResponseFlexible(nn.Module):
    """Swappable drug_encoder / fusion / regression_head. Every trained
    ablation in this notebook (concat-fusion, gene-gating-off, phase-1-only,
    ChemBERTa-only, Morgan-only, few-fusion-heads, unfreeze-depth,
    linear-head) is an instance of this same class with one component
    swapped, so run_stage2_pipeline() below works unmodified for all of them."""
    def __init__(self, backbone, drug_encoder, fusion, regression_head):
        super().__init__()
        self.backbone = backbone
        self.freeze_backbone = True
        self.drug_encoder = drug_encoder
        self.fusion = fusion
        self.regression_head = regression_head
    def forward(self, mrna, cnv, mut, meth, input_ids, attention_mask, morgan_fp):
        if self.freeze_backbone:
            with torch.no_grad(): patient_embed = self.backbone(mrna, cnv, mut, meth)
        else:
            patient_embed = self.backbone(mrna, cnv, mut, meth)
        drug_embed = self.drug_encoder(input_ids, attention_mask, morgan_fp)
        fused, _ = self.fusion(patient_embed, drug_embed)
        return self.regression_head(fused)


def get_module(m): return m.module if isinstance(m, nn.DataParallel) else m
print('All architecture classes defined (incl. flexible drug-response model).')


  4. MODEL ARCHITECTURE
All architecture classes defined (incl. flexible drug-response model).


In [5]:
# ═══════════════════════════════════════════════════════════════════════
#  CELL 5 — TRAINING/EVAL UTILITIES
# ═══════════════════════════════════════════════════════════════════════
section('5. TRAINING UTILITIES')

def run_epoch(model, loader, optimizer, amp_scaler, is_train, scheduler=None, grad_accum=1, split_mode='warm'):
    model.train() if is_train else model.eval()
    criterion = nn.MSELoss()
    tot_loss, n_samp = 0.0, 0
    all_preds, all_tgts, all_dids = [], [], []
    if is_train: optimizer.zero_grad()
    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for step, b in enumerate(loader):
            mrna = b['mrna'].to(DEVICE, non_blocking=True); cnv = b['cnv'].to(DEVICE, non_blocking=True)
            mut  = b['mut'].to(DEVICE, non_blocking=True);  meth = b['meth'].to(DEVICE, non_blocking=True)
            ids  = b['input_ids'].to(DEVICE, non_blocking=True); mask = b['attention_mask'].to(DEVICE, non_blocking=True)
            fp   = b['morgan_fp'].to(DEVICE, non_blocking=True)
            ic50 = b['ic50'].to(DEVICE, non_blocking=True); did = b['drug_idx'].to(DEVICE, non_blocking=True)
            bs = ic50.size(0)
            with torch.cuda.amp.autocast(enabled=CONFIG['use_amp']):
                pred = model(mrna, cnv, mut, meth, ids, mask, fp).squeeze(1)
                loss = criterion(pred, ic50) / grad_accum
            if is_train:
                amp_scaler.scale(loss).backward()
                if (step + 1) % grad_accum == 0:
                    amp_scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['clip_grad'])
                    amp_scaler.step(optimizer); amp_scaler.update()
                    optimizer.zero_grad()
                    if scheduler is not None: scheduler.step()
            tot_loss += loss.item() * grad_accum * bs
            n_samp += bs
            all_preds.append(pred.detach().cpu().numpy())
            all_tgts.append(ic50.detach().cpu().numpy())
            all_dids.append(did.detach().cpu().numpy())
    return tot_loss / n_samp, np.concatenate(all_preds), np.concatenate(all_tgts), np.concatenate(all_dids)


def compute_metrics(preds, targets, drug_ids, id_to_drug, min_drug_samples=5):
    preds = np.array(preds, dtype=np.float64); targets = np.array(targets, dtype=np.float64)
    pearson_r, _  = pearsonr(preds, targets)
    spearman_r, _ = spearmanr(preds, targets)
    rmse = float(np.sqrt(mean_squared_error(targets, preds)))
    r2   = float(r2_score(targets, preds))
    per_drug = {}
    for did in np.unique(drug_ids):
        mask = (drug_ids == did)
        if mask.sum() < min_drug_samples: continue
        dp, dt = preds[mask], targets[mask]
        dname = id_to_drug.get(int(did), f'drug_{did}')
        drug_pcc = float(pearsonr(dp, dt)[0]) if len(np.unique(dt)) > 1 else 0.0
        per_drug[dname] = {'pearson': drug_pcc, 'n_samples': int(mask.sum())}
    pcc_vals = [v['pearson'] for v in per_drug.values()]
    return {'pearson': float(pearson_r), 'spearman': float(spearman_r), 'rmse': rmse, 'r2': r2,
            'avg_drug_pearson': float(np.mean(pcc_vals)) if pcc_vals else 0.0,
            'n_drugs_evaluated': len(per_drug), 'per_drug': per_drug}

def get_ckpt_val(vm, split_mode): return vm['pearson']

print('run_epoch(), compute_metrics(), get_ckpt_val() defined.')
print('NOTE: AUROC / drug-cold intentionally excluded from all reporting in this notebook.')


  5. TRAINING UTILITIES
run_epoch(), compute_metrics(), get_ckpt_val() defined.
NOTE: AUROC / drug-cold intentionally excluded from all reporting in this notebook.


In [6]:
# ═══════════════════════════════════════════════════════════════════════
#  CELL 6 — LOAD EXISTING FULL MODEL (NO TRAINING). Falls back to reported
#  manuscript numbers if the checkpoint isn't found in this session.
# ═══════════════════════════════════════════════════════════════════════
section('6. LOAD EXISTING FULL MODEL')

HAVE_FULL_MODEL_LOADED = False
full_model = None

MANUSCRIPT_REFERENCE = {
    'warm':      {'pearson': 0.9423, 'avg_drug_pearson': 0.2965, 'rmse': 0.9127, 'r2': 0.8815},
    'cell_cold': {'pearson': 0.9368, 'avg_drug_pearson': 0.2200, 'rmse': 0.9294, 'r2': 0.8755},
}

def build_full_drug_encoder():
    return ChemBERTaDrugEncoder(CONFIG['chemberta_model'], CONFIG['chemberta_dim'],
        CONFIG['chemberta_freeze_layers'], CONFIG['fusion_dropout'], CONFIG['morgan_dim'])

def build_full_fusion(num_heads=None):
    return DrugPatientCrossAttention(CONFIG['embed_dim'] * 5, CONFIG['chemberta_dim'],
        CONFIG['embed_dim'], num_heads or CONFIG['fusion_heads'], CONFIG['fusion_dropout'], 512)

try:
    backbone0 = OncoBridgeMMCAT_v7(CONFIG).to(DEVICE)
    ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
    sd = ckpt if not isinstance(ckpt, dict) or 'state_dict' not in ckpt else ckpt['state_dict']
    sd = {k.replace('module.', ''): v for k, v in sd.items()}
    backbone0.load_state_dict(sd, strict=False)
    for p in backbone0.parameters(): p.requires_grad = False
    backbone0.classifier = nn.Identity()

    full_model = OncoBridgeDrugResponseFlexible(
        backbone=backbone0, drug_encoder=build_full_drug_encoder(),
        fusion=build_full_fusion(), regression_head=FullRegressionHead(512, CONFIG['fusion_dropout'])).to(DEVICE)
    full_model.load_state_dict(torch.load(EXISTING_STAGE2_CKPT_PATH, map_location=DEVICE))
    if N_GPUS > 1:
        full_model = nn.DataParallel(full_model)
    full_model.eval()
    HAVE_FULL_MODEL_LOADED = True
    print(f'  Loaded existing fine-tuned checkpoint from {EXISTING_STAGE2_CKPT_PATH}')

    _, te_p, te_t, te_d = run_epoch(full_model, test_loader, None, None, is_train=False, split_mode=SPLIT_MODE)
    full_model_results = compute_metrics(te_p, te_t, te_d, id_to_drug)
    print(f'  [{SPLIT_MODE}] PCC={full_model_results["pearson"]:.4f} | '
          f'AvgDrugPCC={full_model_results["avg_drug_pearson"]:.4f} | '
          f'RMSE={full_model_results["rmse"]:.4f} | R2={full_model_results["r2"]:.4f}')

except FileNotFoundError:
    print(f'  [!] Could not find {EXISTING_STAGE2_CKPT_PATH} in this session.')
    print(f'  Falling back to manuscript-reported numbers for split={SPLIT_MODE!r}.')
    print(f'  Update EXISTING_STAGE2_CKPT_PATH in Cell 1 to reload the real checkpoint --')
    print(f'  add your previous run output as a Kaggle input dataset.')
    full_model_results = dict(MANUSCRIPT_REFERENCE[SPLIT_MODE])
    full_model_results['n_drugs_evaluated'] = None
    print(f'  [reported, not reloaded] PCC={full_model_results["pearson"]:.4f} | '
          f'AvgDrugPCC={full_model_results["avg_drug_pearson"]:.4f}')

print(f'\nHAVE_FULL_MODEL_LOADED = {HAVE_FULL_MODEL_LOADED}')


  6. LOAD EXISTING FULL MODEL


pytorch_model.bin:   0%|          | 0.00/13.7M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/53 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: DeepChem/ChemBERTa-77M-MLM
Key                             | Status     | 
--------------------------------+------------+-
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  ChemBERTa: 3,427,440 total | 0 trainable (layers 6+)
  [!] Could not find /kaggle/working/oncobridge_chemberta_v5_best.pt in this session.
  Falling back to manuscript-reported numbers for split='cell_cold'.
  Update EXISTING_STAGE2_CKPT_PATH in Cell 1 to reload the real checkpoint --
  add your previous run output as a Kaggle input dataset.
  [reported, not reloaded] PCC=0.9368 | AvgDrugPCC=0.2200

HAVE_FULL_MODEL_LOADED = False


## Free lesion tests (Cells 7–8) then trained ablations (Cells 10–17)

Cells 7–8 need no training — skipped automatically if `HAVE_FULL_MODEL_LOADED`
is False. Cell 9 defines the shared training helper. Cells 10–17 each run one
trained ablation at the reduced budget from Cell 1.

In [7]:
# ═══════════════════════════════════════════════════════════════════════
#  ABLATION 1 & 2 — DRUG-IDENTITY LESION + CELL-LINE LESION  (FREE)
# ═══════════════════════════════════════════════════════════════════════
def evaluate_with_lesion(model, loader, tag, lesion=None, zero_modality=None):
    m = get_module(model)
    m.eval()
    all_preds, all_tgts, all_dids = [], [], []
    with torch.no_grad():
        for b in loader:
            mrna = b['mrna'].to(DEVICE); cnv = b['cnv'].to(DEVICE)
            mut  = b['mut'].to(DEVICE);  meth = b['meth'].to(DEVICE)
            ids  = b['input_ids'].to(DEVICE); mask = b['attention_mask'].to(DEVICE)
            fp   = b['morgan_fp'].to(DEVICE)
            ic50 = b['ic50'].to(DEVICE); did = b['drug_idx'].to(DEVICE)
            if zero_modality == 'mrna': mrna = torch.zeros_like(mrna)
            if zero_modality == 'cnv':  cnv  = torch.zeros_like(cnv)
            if zero_modality == 'mut':  mut  = torch.zeros_like(mut)
            if zero_modality == 'meth': meth = torch.zeros_like(meth)
            with torch.cuda.amp.autocast(enabled=CONFIG['use_amp']):
                patient_embed = m.backbone(mrna, cnv, mut, meth)
                drug_embed    = m.drug_encoder(ids, mask, fp)
                if lesion == 'drug_identity':
                    drug_embed = drug_embed.mean(dim=0, keepdim=True).expand_as(drug_embed)
                if lesion == 'cell_line':
                    patient_embed = patient_embed.mean(dim=0, keepdim=True).expand_as(patient_embed)
                fused, _ = m.fusion(patient_embed, drug_embed)
                pred = m.regression_head(fused).squeeze(1)
            all_preds.append(pred.float().cpu().numpy())
            all_tgts.append(ic50.cpu().numpy())
            all_dids.append(did.cpu().numpy())
    res = compute_metrics(np.concatenate(all_preds), np.concatenate(all_tgts),
                           np.concatenate(all_dids), id_to_drug)
    print(f'  [{tag}] PCC={res["pearson"]:.4f} | AvgDrugPCC={res["avg_drug_pearson"]:.4f} | '
          f'RMSE={res["rmse"]:.4f} | R2={res["r2"]:.4f}')
    return res

if HAVE_FULL_MODEL_LOADED:
    print(f'\n{"="*70}\n  LESION TESTS (on loaded full model)\n{"="*70}')
    lesion_drug_identity = evaluate_with_lesion(full_model, test_loader, 'Drug-identity lesion', lesion='drug_identity')
    lesion_cell_line      = evaluate_with_lesion(full_model, test_loader, 'Cell-line lesion',      lesion='cell_line')
else:
    print('Skipping drug-identity / cell-line lesion tests \u2014 no loaded full model available this session.')
    lesion_drug_identity = lesion_cell_line = None

Skipping drug-identity / cell-line lesion tests — no loaded full model available this session.


In [8]:
# ═══════════════════════════════════════════════════════════════════════
#  ABLATION 3 — MODALITY ZERO-OUT LESION  (FREE, ×4)
# ═══════════════════════════════════════════════════════════════════════
modality_lesion_results = {}
if HAVE_FULL_MODEL_LOADED:
    print(f'\n{"="*70}\n  MODALITY ZERO-OUT LESION (on loaded full model)\n{"="*70}')
    for mod in ['mrna', 'cnv', 'mut', 'meth']:
        modality_lesion_results[mod] = evaluate_with_lesion(
            full_model, test_loader, f'Zero-{mod}', zero_modality=mod)
else:
    print('Skipping modality zero-out lesions \u2014 no loaded full model available this session.')

Skipping modality zero-out lesions — no loaded full model available this session.


In [9]:
# ═══════════════════════════════════════════════════════════════════════
#  SHARED HELPER — fresh backbone + self-contained 2-phase pipeline for
#  every trained ablation below. unfreeze_layers lets Ablation 10 override
#  the default without touching CONFIG globally.
# ═══════════════════════════════════════════════════════════════════════
def build_fresh_backbone(disable_gating=False):
    bb = OncoBridgeMMCAT_v7(CONFIG).to(DEVICE)
    ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
    sd = ckpt if not isinstance(ckpt, dict) or 'state_dict' not in ckpt else ckpt['state_dict']
    sd = {k.replace('module.', ''): v for k, v in sd.items()}
    bb.load_state_dict(sd, strict=False)
    for p in bb.parameters(): p.requires_grad = False
    bb.classifier = nn.Identity()
    if disable_gating:
        for enc_name in ['mrna_enc', 'cnv_enc', 'mut_enc', 'meth_enc']:
            getattr(bb, enc_name).gene_gate.forward = lambda x: x
        print('  [ablation] Gene-importance gating DISABLED on all 4 modality encoders.')
    return bb


def run_stage2_pipeline(model, tag, ckpt_path, phase2=True, unfreeze_layers=None):
    if N_GPUS > 1: model = nn.DataParallel(model)
    unfreeze_layers = CONFIG['stage2_unfreeze_mod_layers'] if unfreeze_layers is None else unfreeze_layers

    print(f'\n{"="*70}\n  {tag} \u2014 PHASE 1\n{"="*70}')
    m = get_module(model)
    opt_s1 = optim.AdamW([
        {'params': m.drug_encoder.parameters(), 'lr': CONFIG['stage1_lr'] * 0.1},
        {'params': m.fusion.parameters(),       'lr': CONFIG['stage1_lr']},
        {'params': m.regression_head.parameters(), 'lr': CONFIG['stage1_lr']},
    ], weight_decay=CONFIG['stage1_wd'])
    steps_per_epoch = math.ceil(len(train_loader) / CONFIG['grad_accum'])
    sched_s1 = optim.lr_scheduler.OneCycleLR(
        opt_s1, max_lr=[CONFIG['stage1_lr']*0.1, CONFIG['stage1_lr'], CONFIG['stage1_lr']],
        total_steps=steps_per_epoch * CONFIG['stage1_epochs'], pct_start=CONFIG['warmup_frac'],
        anneal_strategy='cos', div_factor=10.0, final_div_factor=100.0)
    scaler_s1 = torch.cuda.amp.GradScaler(enabled=CONFIG['use_amp'])
    best_s1, patience_s1 = -np.inf, 0

    for epoch in range(1, CONFIG['stage1_epochs'] + 1):
        tr_loss, _, _, _ = run_epoch(model, train_loader, opt_s1, scaler_s1, True,
                                      sched_s1, CONFIG['grad_accum'], split_mode=SPLIT_MODE)
        vl_loss, vl_p, vl_t, vl_d = run_epoch(model, val_loader, opt_s1, scaler_s1, False, split_mode=SPLIT_MODE)
        vm = compute_metrics(vl_p, vl_t, vl_d, id_to_drug)
        print(f'  [P1 {epoch:02d}/{CONFIG["stage1_epochs"]}] TrLoss={tr_loss:.4f} '
              f'PCC={vm["pearson"]:.4f} DrugPCC={vm["avg_drug_pearson"]:.4f}')
        ckpt_val = get_ckpt_val(vm, SPLIT_MODE)
        if ckpt_val > best_s1:
            best_s1, patience_s1 = ckpt_val, 0
            torch.save(get_module(model).state_dict(), ckpt_path)
            print(f'    \u2713 Best saved  PCC={best_s1:.4f}')
        else:
            patience_s1 += 1
            if patience_s1 >= CONFIG['stage1_patience']:
                print(f'    Early stop at epoch {epoch}'); break
    print(f'  Phase 1 complete. Best PCC = {best_s1:.4f}')

    if not phase2:
        get_module(model).load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
        model.eval()
        _, te_p, te_t, te_d = run_epoch(model, test_loader, None, None, is_train=False, split_mode=SPLIT_MODE)
        res = compute_metrics(te_p, te_t, te_d, id_to_drug)
        print(f'\n  \u2605 {tag} \u2014 {SPLIT_MODE.upper()} TEST (Phase 1 only)')
        print(f'    PCC={res["pearson"]:.4f} | AvgDrugPCC={res["avg_drug_pearson"]:.4f} | '
              f'RMSE={res["rmse"]:.4f} | R2={res["r2"]:.4f}')
        return res

    print(f'\n{"="*70}\n  {tag} \u2014 PHASE 2 (unfreeze_layers={unfreeze_layers})\n{"="*70}')
    get_module(model).load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    get_module(model).freeze_backbone = False
    bm = get_module(model).backbone
    for p in bm.cross_layers.parameters(): p.requires_grad = True
    for p in bm.fusion.parameters():       p.requires_grad = True
    modality_enc_params = []
    if unfreeze_layers > 0:
        for enc_name in ['mrna_enc', 'cnv_enc', 'mut_enc', 'meth_enc']:
            enc = getattr(bm, enc_name)
            n_layers = len(enc.transformer.layers)
            for i, layer in enumerate(enc.transformer.layers):
                if i >= n_layers - unfreeze_layers:
                    for p in layer.parameters(): p.requires_grad = True
                    modality_enc_params.extend(list(layer.parameters()))
            for p in enc.norm.parameters(): p.requires_grad = True
            modality_enc_params.extend(list(enc.norm.parameters()))
    seen, unique_mod_params = set(), []
    for p in modality_enc_params:
        if id(p) not in seen: seen.add(id(p)); unique_mod_params.append(p)

    m = get_module(model)
    param_groups = [
        {'params': list(m.regression_head.parameters()), 'lr': CONFIG['stage2_lr_head']},
        {'params': list(m.fusion.parameters()),           'lr': CONFIG['stage2_lr_head']},
        {'params': list(bm.cross_layers.parameters()),    'lr': CONFIG['stage2_lr_cross']},
        {'params': list(bm.fusion.parameters()),          'lr': CONFIG['stage2_lr_cross']},
    ]
    max_lrs = [CONFIG['stage2_lr_head'], CONFIG['stage2_lr_head'],
               CONFIG['stage2_lr_cross'], CONFIG['stage2_lr_cross']]
    if hasattr(m.drug_encoder, 'encoder'):
        param_groups.append({'params': [p for p in m.drug_encoder.encoder.parameters() if p.requires_grad],
                              'lr': CONFIG['stage2_lr_cbert']})
        max_lrs.append(CONFIG['stage2_lr_cbert'])
        proj_params = []
        for attr in ['smiles_proj', 'morgan_proj', 'fusion_proj']:
            if hasattr(m.drug_encoder, attr):
                proj_params += list(getattr(m.drug_encoder, attr).parameters())
        param_groups.append({'params': proj_params, 'lr': CONFIG['stage2_lr_proj']})
        max_lrs.append(CONFIG['stage2_lr_proj'])
    else:
        # Morgan-only: just one projection block, no ChemBERTa
        param_groups.append({'params': list(m.drug_encoder.parameters()), 'lr': CONFIG['stage2_lr_proj']})
        max_lrs.append(CONFIG['stage2_lr_proj'])
    if unique_mod_params:
        param_groups.append({'params': unique_mod_params, 'lr': CONFIG['stage2_lr_mod_enc']})
        max_lrs.append(CONFIG['stage2_lr_mod_enc'])

    opt_s2 = optim.AdamW(param_groups, weight_decay=CONFIG['stage2_wd'])
    total_steps_s2 = math.ceil(len(train_loader) / CONFIG['grad_accum']) * CONFIG['stage2_epochs']
    sched_s2 = optim.lr_scheduler.OneCycleLR(
        opt_s2, max_lr=max_lrs, total_steps=total_steps_s2, pct_start=CONFIG['warmup_frac'],
        anneal_strategy='cos', div_factor=10.0, final_div_factor=100.0)
    scaler_s2 = torch.cuda.amp.GradScaler(enabled=CONFIG['use_amp'])
    best_s2, patience_s2 = -np.inf, 0

    for epoch in range(1, CONFIG['stage2_epochs'] + 1):
        tr_loss, _, _, _ = run_epoch(model, train_loader, opt_s2, scaler_s2, True,
                                      sched_s2, CONFIG['grad_accum'], split_mode=SPLIT_MODE)
        vl_loss, vl_p, vl_t, vl_d = run_epoch(model, val_loader, opt_s2, scaler_s2, False, split_mode=SPLIT_MODE)
        vm = compute_metrics(vl_p, vl_t, vl_d, id_to_drug)
        print(f'  [P2 {epoch:02d}/{CONFIG["stage2_epochs"]}] TrLoss={tr_loss:.4f} '
              f'PCC={vm["pearson"]:.4f} DrugPCC={vm["avg_drug_pearson"]:.4f}')
        ckpt_val = get_ckpt_val(vm, SPLIT_MODE)
        if ckpt_val > best_s2:
            best_s2, patience_s2 = ckpt_val, 0
            torch.save(get_module(model).state_dict(), ckpt_path)
            print(f'    \u2713 Best saved  PCC={best_s2:.4f}')
        else:
            patience_s2 += 1
            if patience_s2 >= CONFIG['stage2_patience']:
                print(f'    Early stop at epoch {epoch}'); break
    print(f'  Phase 2 complete. Best PCC = {best_s2:.4f}')

    get_module(model).load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    model.eval()
    _, te_p, te_t, te_d = run_epoch(model, test_loader, None, None, is_train=False, split_mode=SPLIT_MODE)
    res = compute_metrics(te_p, te_t, te_d, id_to_drug)
    print(f'\n  \u2605 {tag} \u2014 {SPLIT_MODE.upper()} TEST SET')
    print(f'    PCC={res["pearson"]:.4f} | AvgDrugPCC={res["avg_drug_pearson"]:.4f} | '
          f'RMSE={res["rmse"]:.4f} | R2={res["r2"]:.4f}')
    return res

print('build_fresh_backbone() and run_stage2_pipeline() defined.')

build_fresh_backbone() and run_stage2_pipeline() defined.


In [10]:
# ═══════════════════════════════════════════════════════════════════════
#  ABLATION 4 — CONCAT FUSION
# ═══════════════════════════════════════════════════════════════════════
backbone_concat = build_fresh_backbone(disable_gating=False)
model_concat = OncoBridgeDrugResponseFlexible(
    backbone=backbone_concat, drug_encoder=build_full_drug_encoder(),
    fusion=DrugPatientConcatFusion(CONFIG['embed_dim']*5, CONFIG['chemberta_dim'], 512, CONFIG['fusion_dropout']),
    regression_head=FullRegressionHead(512, CONFIG['fusion_dropout'])).to(DEVICE)
concat_results = run_stage2_pipeline(model_concat, 'CONCAT FUSION (no cross-attn)',
                                      f'{OUT_DIR}oncobridge_concat_fusion_{SPLIT_MODE}_best.pt')

model.safetensors:   0%|          | 0.00/13.7M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/53 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: DeepChem/ChemBERTa-77M-MLM
Key                             | Status     | 
--------------------------------+------------+-
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  ChemBERTa: 3,427,440 total | 0 trainable (layers 6+)

  CONCAT FUSION (no cross-attn) — PHASE 1
  [P1 01/8] TrLoss=4.9456 PCC=0.8594 DrugPCC=-0.1071
    ✓ Best saved  PCC=0.8594
  [P1 02/8] TrLoss=1.5280 PCC=0.8999 DrugPCC=0.1584
    ✓ Best saved  PCC=0.8999
  [P1 03/8] TrLoss=1.1902 PCC=0.9066 DrugPCC=0.1103
    ✓ Best saved  PCC=0.9066
  [P1 04/8] TrLoss=1.0279 PCC=0.9092 DrugPCC=0.1472
    ✓ Best saved  PCC=0.9092
  [P1 05/8] TrLoss=0.9524 PCC=0.9132 DrugPCC=0.1487
    ✓ Best saved  PCC=0.9132
  [P1 06/8] TrLoss=0.9336 PCC=0.9139 DrugPCC=0.2009
    ✓ Best saved  PCC=0.9139
  [P1 07/8] TrLoss=0.8855 PCC=0.9141 DrugPCC=0.1710
    ✓ Best saved  PCC=0.9141
  [P1 08/8] TrLoss=0.8514 PCC=0.9136 DrugPCC=0.1544
  Phase 1 complete. Best PCC = 0.9141

  CONCAT FUSION (no cross-attn) — PHASE 2 (unfreeze_layers=2)
  [P2 01/10] TrLoss=0.8662 PCC=0.9143 DrugPCC=0.1546
    ✓ Best saved  PCC=0.9143
  [P2 02/10] TrLoss=0.9039 PCC=0.9128 DrugPCC=0.1778
  [P2 03/10] TrLoss=0.9008 PCC=0.9159 DrugPCC=

In [11]:
# ═══════════════════════════════════════════════════════════════════════
#  ABLATION 5 — GENE-GATING DISABLED
# ═══════════════════════════════════════════════════════════════════════
backbone_nogate = build_fresh_backbone(disable_gating=True)
model_nogate = OncoBridgeDrugResponseFlexible(
    backbone=backbone_nogate, drug_encoder=build_full_drug_encoder(),
    fusion=build_full_fusion(), regression_head=FullRegressionHead(512, CONFIG['fusion_dropout'])).to(DEVICE)
nogate_results = run_stage2_pipeline(model_nogate, 'GENE-GATING DISABLED',
                                      f'{OUT_DIR}oncobridge_nogate_{SPLIT_MODE}_best.pt')

  [ablation] Gene-importance gating DISABLED on all 4 modality encoders.


Loading weights:   0%|          | 0/53 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: DeepChem/ChemBERTa-77M-MLM
Key                             | Status     | 
--------------------------------+------------+-
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  ChemBERTa: 3,427,440 total | 0 trainable (layers 6+)

  GENE-GATING DISABLED — PHASE 1
  [P1 01/8] TrLoss=3.7681 PCC=0.8906 DrugPCC=0.0662
    ✓ Best saved  PCC=0.8906
  [P1 02/8] TrLoss=1.2589 PCC=0.8986 DrugPCC=0.0234
    ✓ Best saved  PCC=0.8986
  [P1 03/8] TrLoss=1.0992 PCC=0.9113 DrugPCC=0.1302
    ✓ Best saved  PCC=0.9113
  [P1 04/8] TrLoss=1.0180 PCC=0.9081 DrugPCC=0.1509
  [P1 05/8] TrLoss=0.9416 PCC=0.9143 DrugPCC=0.1915
    ✓ Best saved  PCC=0.9143
  [P1 06/8] TrLoss=0.8485 PCC=0.9152 DrugPCC=0.1688
    ✓ Best saved  PCC=0.9152
  [P1 07/8] TrLoss=0.7974 PCC=0.9150 DrugPCC=0.1619
  [P1 08/8] TrLoss=0.8003 PCC=0.9150 DrugPCC=0.1552
  Phase 1 complete. Best PCC = 0.9152

  GENE-GATING DISABLED — PHASE 2 (unfreeze_layers=2)
  [P2 01/10] TrLoss=0.8349 PCC=0.9180 DrugPCC=0.2496
    ✓ Best saved  PCC=0.9180
  [P2 02/10] TrLoss=0.8117 PCC=0.9123 DrugPCC=0.1771
  [P2 03/10] TrLoss=0.7825 PCC=0.9153 DrugPCC=0.1792
  [P2 04/10] TrLoss=0.7457 PCC=0.9141 DrugPCC=0.1777
  [P2 05/10] TrLo

In [12]:
# ═══════════════════════════════════════════════════════════════════════
#  ABLATION 6 — PHASE-1-ONLY (skip Phase 2 entirely)
# ═══════════════════════════════════════════════════════════════════════
backbone_p1 = build_fresh_backbone(disable_gating=False)
model_p1 = OncoBridgeDrugResponseFlexible(
    backbone=backbone_p1, drug_encoder=build_full_drug_encoder(),
    fusion=build_full_fusion(), regression_head=FullRegressionHead(512, CONFIG['fusion_dropout'])).to(DEVICE)
phase1_only_results = run_stage2_pipeline(model_p1, 'PHASE-1-ONLY (frozen backbone throughout)',
                                           f'{OUT_DIR}oncobridge_phase1only_{SPLIT_MODE}_best.pt',
                                           phase2=False)

Loading weights:   0%|          | 0/53 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: DeepChem/ChemBERTa-77M-MLM
Key                             | Status     | 
--------------------------------+------------+-
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  ChemBERTa: 3,427,440 total | 0 trainable (layers 6+)

  PHASE-1-ONLY (frozen backbone throughout) — PHASE 1
  [P1 01/8] TrLoss=3.7259 PCC=0.8836 DrugPCC=0.1586
    ✓ Best saved  PCC=0.8836
  [P1 02/8] TrLoss=1.2777 PCC=0.9047 DrugPCC=0.2089
    ✓ Best saved  PCC=0.9047
  [P1 03/8] TrLoss=1.0743 PCC=0.9049 DrugPCC=0.1360
    ✓ Best saved  PCC=0.9049
  [P1 04/8] TrLoss=1.0067 PCC=0.9112 DrugPCC=0.1746
    ✓ Best saved  PCC=0.9112
  [P1 05/8] TrLoss=0.9150 PCC=0.9145 DrugPCC=0.1726
    ✓ Best saved  PCC=0.9145
  [P1 06/8] TrLoss=0.8705 PCC=0.9159 DrugPCC=0.1956
    ✓ Best saved  PCC=0.9159
  [P1 07/8] TrLoss=0.8031 PCC=0.9156 DrugPCC=0.2264
  [P1 08/8] TrLoss=0.7682 PCC=0.9155 DrugPCC=0.2274
  Phase 1 complete. Best PCC = 0.9159

  ★ PHASE-1-ONLY (frozen backbone throughout) — CELL_COLD TEST (Phase 1 only)
    PCC=0.9408 | AvgDrugPCC=0.2057 | RMSE=0.9201 | R2=0.8779


In [13]:
# ═══════════════════════════════════════════════════════════════════════
#  ABLATION 7 — CHEMBERTA-ONLY DRUG ENCODER (no Morgan FP branch)
# ═══════════════════════════════════════════════════════════════════════
backbone_cb = build_fresh_backbone(disable_gating=False)
model_cbonly = OncoBridgeDrugResponseFlexible(
    backbone=backbone_cb,
    drug_encoder=ChemBERTaOnlyDrugEncoder(CONFIG['chemberta_model'], CONFIG['chemberta_dim'],
                                           CONFIG['chemberta_freeze_layers'], CONFIG['fusion_dropout']),
    fusion=build_full_fusion(), regression_head=FullRegressionHead(512, CONFIG['fusion_dropout'])).to(DEVICE)
chemberta_only_results = run_stage2_pipeline(model_cbonly, 'CHEMBERTA-ONLY (no Morgan FP)',
                                              f'{OUT_DIR}oncobridge_cbonly_{SPLIT_MODE}_best.pt')

Loading weights:   0%|          | 0/53 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: DeepChem/ChemBERTa-77M-MLM
Key                             | Status     | 
--------------------------------+------------+-
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



  CHEMBERTA-ONLY (no Morgan FP) — PHASE 1
  [P1 01/8] TrLoss=4.6890 PCC=0.8118 DrugPCC=0.0456
    ✓ Best saved  PCC=0.8118
  [P1 02/8] TrLoss=2.5332 PCC=0.8588 DrugPCC=-0.0305
    ✓ Best saved  PCC=0.8588
  [P1 03/8] TrLoss=1.9544 PCC=0.8807 DrugPCC=-0.0017
    ✓ Best saved  PCC=0.8807
  [P1 04/8] TrLoss=1.6891 PCC=0.8903 DrugPCC=0.0534
    ✓ Best saved  PCC=0.8903
  [P1 05/8] TrLoss=1.4578 PCC=0.9003 DrugPCC=0.1144
    ✓ Best saved  PCC=0.9003
  [P1 06/8] TrLoss=1.3523 PCC=0.9023 DrugPCC=0.1717
    ✓ Best saved  PCC=0.9023
  [P1 07/8] TrLoss=1.2742 PCC=0.9022 DrugPCC=0.1460
  [P1 08/8] TrLoss=1.2182 PCC=0.9030 DrugPCC=0.1530
    ✓ Best saved  PCC=0.9030
  Phase 1 complete. Best PCC = 0.9030

  CHEMBERTA-ONLY (no Morgan FP) — PHASE 2 (unfreeze_layers=2)
  [P2 01/10] TrLoss=1.2300 PCC=0.8980 DrugPCC=0.0923
    ✓ Best saved  PCC=0.8980
  [P2 02/10] TrLoss=1.2079 PCC=0.9040 DrugPCC=0.1928
    ✓ Best saved  PCC=0.9040
  [P2 03/10] TrLoss=1.1948 PCC=0.9015 DrugPCC=0.2105
  [P2 04/10] TrLos

In [14]:
# ═══════════════════════════════════════════════════════════════════════
#  ABLATION 8 — MORGAN-ONLY DRUG ENCODER (no ChemBERTa — cheapest run here)
# ═══════════════════════════════════════════════════════════════════════
backbone_mg = build_fresh_backbone(disable_gating=False)
model_mgonly = OncoBridgeDrugResponseFlexible(
    backbone=backbone_mg,
    drug_encoder=MorganOnlyDrugEncoder(CONFIG['chemberta_dim'], CONFIG['morgan_dim'], CONFIG['fusion_dropout']),
    fusion=build_full_fusion(), regression_head=FullRegressionHead(512, CONFIG['fusion_dropout'])).to(DEVICE)
morgan_only_results = run_stage2_pipeline(model_mgonly, 'MORGAN-ONLY (no ChemBERTa)',
                                           f'{OUT_DIR}oncobridge_mgonly_{SPLIT_MODE}_best.pt')


  MORGAN-ONLY (no ChemBERTa) — PHASE 1
  [P1 01/8] TrLoss=3.5251 PCC=0.9017 DrugPCC=0.1785
    ✓ Best saved  PCC=0.9017
  [P1 02/8] TrLoss=1.1281 PCC=0.9078 DrugPCC=0.1392
    ✓ Best saved  PCC=0.9078
  [P1 03/8] TrLoss=1.0392 PCC=0.9051 DrugPCC=0.0799
  [P1 04/8] TrLoss=0.9730 PCC=0.9098 DrugPCC=0.1416
    ✓ Best saved  PCC=0.9098
  [P1 05/8] TrLoss=0.8786 PCC=0.9131 DrugPCC=0.2095
    ✓ Best saved  PCC=0.9131
  [P1 06/8] TrLoss=0.8112 PCC=0.9157 DrugPCC=0.1776
    ✓ Best saved  PCC=0.9157
  [P1 07/8] TrLoss=0.7702 PCC=0.9153 DrugPCC=0.1744
  [P1 08/8] TrLoss=0.7094 PCC=0.9154 DrugPCC=0.1801
  Phase 1 complete. Best PCC = 0.9157

  MORGAN-ONLY (no ChemBERTa) — PHASE 2 (unfreeze_layers=2)
  [P2 01/10] TrLoss=0.7719 PCC=0.9129 DrugPCC=0.1599
    ✓ Best saved  PCC=0.9129
  [P2 02/10] TrLoss=0.7594 PCC=0.9136 DrugPCC=0.2021
    ✓ Best saved  PCC=0.9136
  [P2 03/10] TrLoss=0.7085 PCC=0.9159 DrugPCC=0.1833
    ✓ Best saved  PCC=0.9159
  [P2 04/10] TrLoss=0.6551 PCC=0.9123 DrugPCC=0.2737
  

In [15]:
# ═══════════════════════════════════════════════════════════════════════
#  ABLATION 9 — FEW FUSION-HEADS (8 -> 2)
# ═══════════════════════════════════════════════════════════════════════
backbone_fh = build_fresh_backbone(disable_gating=False)
model_fewheads = OncoBridgeDrugResponseFlexible(
    backbone=backbone_fh, drug_encoder=build_full_drug_encoder(),
    fusion=build_full_fusion(num_heads=2), regression_head=FullRegressionHead(512, CONFIG['fusion_dropout'])).to(DEVICE)
fewheads_results = run_stage2_pipeline(model_fewheads, 'FEW FUSION-HEADS (2, was 8)',
                                        f'{OUT_DIR}oncobridge_fewheads_{SPLIT_MODE}_best.pt')

Loading weights:   0%|          | 0/53 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: DeepChem/ChemBERTa-77M-MLM
Key                             | Status     | 
--------------------------------+------------+-
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  ChemBERTa: 3,427,440 total | 0 trainable (layers 6+)

  FEW FUSION-HEADS (2, was 8) — PHASE 1
  [P1 01/8] TrLoss=3.6585 PCC=0.8863 DrugPCC=0.0662
    ✓ Best saved  PCC=0.8863
  [P1 02/8] TrLoss=1.3142 PCC=0.9083 DrugPCC=0.1296
    ✓ Best saved  PCC=0.9083
  [P1 03/8] TrLoss=1.1018 PCC=0.9000 DrugPCC=0.1174
  [P1 04/8] TrLoss=1.0245 PCC=0.9124 DrugPCC=0.1093
    ✓ Best saved  PCC=0.9124
  [P1 05/8] TrLoss=0.9887 PCC=0.9109 DrugPCC=0.1694
  [P1 06/8] TrLoss=0.9137 PCC=0.9113 DrugPCC=0.1225
  [P1 07/8] TrLoss=0.8604 PCC=0.9143 DrugPCC=0.1446
    ✓ Best saved  PCC=0.9143
  [P1 08/8] TrLoss=0.8456 PCC=0.9143 DrugPCC=0.1460
    ✓ Best saved  PCC=0.9143
  Phase 1 complete. Best PCC = 0.9143

  FEW FUSION-HEADS (2, was 8) — PHASE 2 (unfreeze_layers=2)
  [P2 01/10] TrLoss=0.8817 PCC=0.9117 DrugPCC=0.1109
    ✓ Best saved  PCC=0.9117
  [P2 02/10] TrLoss=0.8994 PCC=0.9156 DrugPCC=0.2116
    ✓ Best saved  PCC=0.9156
  [P2 03/10] TrLoss=0.8377 PCC=0.9113 DrugPCC=0.1452
  [P2 04/10] TrLoss=0.8040 

In [16]:
# ═══════════════════════════════════════════════════════════════════════
#  ABLATION 10 — UNFREEZE-DEPTH = 4 (was 2)
# ═══════════════════════════════════════════════════════════════════════
backbone_uf4 = build_fresh_backbone(disable_gating=False)
model_uf4 = OncoBridgeDrugResponseFlexible(
    backbone=backbone_uf4, drug_encoder=build_full_drug_encoder(),
    fusion=build_full_fusion(), regression_head=FullRegressionHead(512, CONFIG['fusion_dropout'])).to(DEVICE)
unfreeze4_results = run_stage2_pipeline(model_uf4, 'UNFREEZE-DEPTH=4 (was 2)',
                                         f'{OUT_DIR}oncobridge_unfreeze4_{SPLIT_MODE}_best.pt',
                                         unfreeze_layers=4)

Loading weights:   0%|          | 0/53 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: DeepChem/ChemBERTa-77M-MLM
Key                             | Status     | 
--------------------------------+------------+-
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  ChemBERTa: 3,427,440 total | 0 trainable (layers 6+)

  UNFREEZE-DEPTH=4 (was 2) — PHASE 1
  [P1 01/8] TrLoss=3.8080 PCC=0.8942 DrugPCC=0.0717
    ✓ Best saved  PCC=0.8942
  [P1 02/8] TrLoss=1.2445 PCC=0.9083 DrugPCC=0.2115
    ✓ Best saved  PCC=0.9083
  [P1 03/8] TrLoss=1.1052 PCC=0.9105 DrugPCC=0.1688
    ✓ Best saved  PCC=0.9105
  [P1 04/8] TrLoss=1.0143 PCC=0.9113 DrugPCC=0.1769
    ✓ Best saved  PCC=0.9113
  [P1 05/8] TrLoss=0.9172 PCC=0.9108 DrugPCC=0.1483
  [P1 06/8] TrLoss=0.8587 PCC=0.9159 DrugPCC=0.1812
    ✓ Best saved  PCC=0.9159
  [P1 07/8] TrLoss=0.8046 PCC=0.9168 DrugPCC=0.1726
    ✓ Best saved  PCC=0.9168
  [P1 08/8] TrLoss=0.7778 PCC=0.9172 DrugPCC=0.1769
    ✓ Best saved  PCC=0.9172
  Phase 1 complete. Best PCC = 0.9172

  UNFREEZE-DEPTH=4 (was 2) — PHASE 2 (unfreeze_layers=4)
  [P2 01/10] TrLoss=0.7963 PCC=0.9130 DrugPCC=0.1847
    ✓ Best saved  PCC=0.9130
  [P2 02/10] TrLoss=0.8211 PCC=0.9130 DrugPCC=0.1795
    ✓ Best saved  PCC=0.9130
  [P2 03/10] TrLoss=0.7641 P

In [17]:
# ═══════════════════════════════════════════════════════════════════════
#  ABLATION 11 — LINEAR REGRESSION HEAD (no depth)
# ═══════════════════════════════════════════════════════════════════════
backbone_lin = build_fresh_backbone(disable_gating=False)
model_linear = OncoBridgeDrugResponseFlexible(
    backbone=backbone_lin, drug_encoder=build_full_drug_encoder(),
    fusion=build_full_fusion(), regression_head=LinearRegressionHead(512)).to(DEVICE)
linear_head_results = run_stage2_pipeline(model_linear, 'LINEAR REGRESSION HEAD (no depth)',
                                           f'{OUT_DIR}oncobridge_linearhead_{SPLIT_MODE}_best.pt')

Loading weights:   0%|          | 0/53 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: DeepChem/ChemBERTa-77M-MLM
Key                             | Status     | 
--------------------------------+------------+-
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  ChemBERTa: 3,427,440 total | 0 trainable (layers 6+)

  LINEAR REGRESSION HEAD (no depth) — PHASE 1
  [P1 01/8] TrLoss=2.5761 PCC=0.8911 DrugPCC=-0.0097
    ✓ Best saved  PCC=0.8911
  [P1 02/8] TrLoss=1.1916 PCC=0.9075 DrugPCC=0.1216
    ✓ Best saved  PCC=0.9075
  [P1 03/8] TrLoss=1.0428 PCC=0.9099 DrugPCC=0.1305
    ✓ Best saved  PCC=0.9099
  [P1 04/8] TrLoss=0.9440 PCC=0.9121 DrugPCC=0.1498
    ✓ Best saved  PCC=0.9121
  [P1 05/8] TrLoss=0.8356 PCC=0.9146 DrugPCC=0.1734
    ✓ Best saved  PCC=0.9146
  [P1 06/8] TrLoss=0.7743 PCC=0.9112 DrugPCC=0.1012
  [P1 07/8] TrLoss=0.7126 PCC=0.9144 DrugPCC=0.1297
  [P1 08/8] TrLoss=0.6951 PCC=0.9152 DrugPCC=0.1322
    ✓ Best saved  PCC=0.9152
  Phase 1 complete. Best PCC = 0.9152

  LINEAR REGRESSION HEAD (no depth) — PHASE 2 (unfreeze_layers=2)
  [P2 01/10] TrLoss=0.7081 PCC=0.9110 DrugPCC=0.1814
    ✓ Best saved  PCC=0.9110
  [P2 02/10] TrLoss=0.7160 PCC=0.9138 DrugPCC=0.2241
    ✓ Best saved  PCC=0.9138
  [P2 03/10] TrLoss=0.6669 PCC=0.9117 

In [18]:
# ═══════════════════════════════════════════════════════════════════════
#  SUMMARY — Stage 2 full ablation suite
# ═══════════════════════════════════════════════════════════════════════
def _row(tag, r):
    print(f'{tag:<44}{r["pearson"]:>9.4f}{r["avg_drug_pearson"]:>13.4f}{r["rmse"]:>9.4f}{r["r2"]:>9.4f}')

print(f'\n{"="*84}')
print(f'  STAGE 2 \u2014 FULL ABLATION SUITE  [{SPLIT_MODE.upper()} SPLIT]')
print(f'{"="*84}')
print(f'{"Model":<44}{"PCC":>9}{"AvgDrugPCC":>13}{"RMSE":>9}{"R2":>9}')
print(f'{"-"*84}')
tag0 = 'Full model' + ('' if HAVE_FULL_MODEL_LOADED else ' [reported, not reloaded]')
_row(tag0, full_model_results)
print(f'{"-"*84}')
if HAVE_FULL_MODEL_LOADED:
    print('  Free lesion tests (forward pass only, no retraining):')
    _row('  Drug-identity lesion', lesion_drug_identity)
    _row('  Cell-line lesion',      lesion_cell_line)
    for mod in ['mrna', 'cnv', 'mut', 'meth']:
        _row(f'  Zero-{mod}', modality_lesion_results[mod])
    print(f'{"-"*84}')
print(f'  Trained ablations (reduced budget: stage1={CONFIG["stage1_epochs"]}ep, '
      f'stage2={CONFIG["stage2_epochs"]}ep \u2014 compare these to EACH OTHER, not to the')
print(f'  full-budget reference above):')
_row('  Concat Fusion (no cross-attn)',   concat_results)
_row('  Gene-Gating Disabled',            nogate_results)
_row('  Phase-1-only (no Phase 2)',       phase1_only_results)
_row('  ChemBERTa-only (no Morgan)',      chemberta_only_results)
_row('  Morgan-only (no ChemBERTa)',      morgan_only_results)
_row('  Few Fusion-Heads (2, was 8)',     fewheads_results)
_row('  Unfreeze-Depth=4 (was 2)',        unfreeze4_results)
_row('  Linear Regression Head',          linear_head_results)
print(f'{"="*84}')
print('\nInterpretation:')
print('  Full vs Drug-identity/Cell-line lesion -> drug-mean vs cell-line-specific PCC')
print('  Full vs Zero-X                         -> which omics layer the model leans on')
print('  Concat vs Gene-Gating vs Phase-1-only   -> already-established ablations')
print('  ChemBERTa-only vs Morgan-only            -> which drug-encoder branch matters')
print('  Few Fusion-Heads                         -> is fusion attention over-provisioned')
print('  Unfreeze-Depth=4 vs default(2)            -> is 2 the right amount of fine-tuning')
print('  Linear Head vs default 3-layer            -> does regression-head depth matter')


  STAGE 2 — FULL ABLATION SUITE  [CELL_COLD SPLIT]
Model                                             PCC   AvgDrugPCC     RMSE       R2
------------------------------------------------------------------------------------
Full model [reported, not reloaded]            0.9368       0.2200   0.9294   0.8755
------------------------------------------------------------------------------------
  Trained ablations (reduced budget: stage1=8ep, stage2=10ep — compare these to EACH OTHER, not to the
  full-budget reference above):
  Concat Fusion (no cross-attn)                0.9427       0.2616   0.9021   0.8827
  Gene-Gating Disabled                         0.9407       0.2603   0.8949   0.8845
  Phase-1-only (no Phase 2)                    0.9408       0.2057   0.9201   0.8779
  ChemBERTa-only (no Morgan)                   0.9335       0.1672   0.9572   0.8679
  Morgan-only (no ChemBERTa)                   0.9346       0.2113   0.9549   0.8685
  Few Fusion-Heads (2, was 8)                  0